In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
load_dotenv()

C:\Users\ansha\AppData\Local\Temp\ipykernel_27824\2137485568.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [ ]:
llm=ChatGroq(model='openai/gpt-oss-120b')   

In [3]:
pdf_name = "Resume(a).pdf"
pdf_candidates = [
    Path.cwd() / pdf_name,
    Path.cwd().parent / pdf_name,
    Path.cwd() / "Rag in langgraph" / pdf_name,
]
pdf_path = next((path.resolve() for path in pdf_candidates if path.is_file()), None)
if pdf_path is None:
    checked_paths = ", ".join(str(path.resolve()) for path in pdf_candidates)
    raise FileNotFoundError(f"Could not find {pdf_name}. Checked: {checked_paths}")

loader = PyPDFLoader(str(pdf_path))
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [4]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(chunks, embeddings)

In [5]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [6]:
@tool
def rag_tool(query: str) -> dict[str, object]:

    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [7]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [8]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def chat_node(state: ChatState):

    messages = [
        SystemMessage(
            content=(
                "You answer using the supplied PDF only. Call rag_tool before answering. "
                "If the retrieved text does not answer the question, say the PDF does not "
                "cover it; do not make up information or use outside knowledge."
            )
        ),
        *state["messages"],
    ]

    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}

In [9]:
tool_node = ToolNode(tools)


In [10]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()

In [13]:
result = chatbot.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Using the pdf notes, eplain if the candidate is good for consulting role be brutally honest"
                )
            )
        ]
    }
)

print(result["messages"][-1].content)

**Candidate Assessment – Consulting Fit (Based solely on the PDF resume)**  

---

### What the resume **does** show  

| Area | Evidence from the PDF |
|------|-----------------------|
| **Education** | B.Tech in Computer Science (IIIT Vadodara) – CGPA 7.5. |
| **Technical foundation** | Data Structures & Algorithms (700+ problems solved), OOP, OS, DBMS. Languages: Java, JavaScript/TypeScript, Python, C/C++, SQL. |
| **Frameworks & tools** | React, Next.js, Node, Express, Tailwind, Socket.IO, MongoDB, MySQL, vector DBs, Git, Redis, Terraform, etc. |
| **Professional experience** | *Backend Developer Intern* at Mentora (Sep 2026‑Present). Built a WebRTC video‑calling service, real‑time chat system, integrated Razorpay digital‑wallet, and added auth/authorization middleware with performance tuning. |
| **Projects** | • **WRDL** – real‑time multiplayer word‑game (full‑stack, Socket.IO, MongoDB). <br>• **FileFlow** – high‑performance file‑transfer engine (real‑time signalling, WebRTC). |
